# Select G7 + China Datasets from Stanford AI Index 2026

This notebook scans all CSV files under `data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/` and selects datasets that contain at least 4 of the G7 + China countries.

**G7 countries**: United States, United Kingdom, France, Germany, Italy, Japan, Canada
**Plus**: China


## 1. Setup

In [1]:
from pathlib import Path
import sys

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'data').is_dir() and (p / 'src').is_dir()
)
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd
import re

from common import find_repo_root

repo_root = find_repo_root(Path.cwd())
data_root = repo_root / 'data' / 'number' / 'PUBLIC DATA_ 2026 AI INDEX REPORT'

print(f'Repo root: {repo_root}')
print(f'Data root: {data_root}')
print(f'Data root exists: {data_root.exists()}')



Repo root: /home/wucheng/AI-Policy
Data root: /home/wucheng/AI-Policy/data/number/PUBLIC DATA_ 2026 AI INDEX REPORT
Data root exists: True


## 2. Define Target Countries

We use full and short names because the Stanford dataset uses both forms (e.g. "United States" vs "United States of America").


In [2]:
from common import TARGET_COUNTRIES

# Minimum number of target countries a file must contain to be included
MIN_COUNTRY_COUNT = 4

print(f'Target countries: {list(TARGET_COUNTRIES.keys())}')
print(f'Minimum match: {MIN_COUNTRY_COUNT} countries')



Target countries: ['United States', 'China', 'United Kingdom', 'France', 'Germany', 'Italy', 'Japan', 'Canada']
Minimum match: 4 countries


## 3. Helper Functions

In [3]:
from data_utils import find_countries_in_file, get_file_metadata, read_csv_safely

print('Helper functions loaded')



Helper functions loaded


## 4. Scan All CSV Files

In [4]:
records = []

for csv_path in sorted(data_root.glob('*/Data/*.csv')):
    countries_found = find_countries_in_file(csv_path)
    
    if len(countries_found) >= MIN_COUNTRY_COUNT:
        meta = get_file_metadata(csv_path)
        records.append({
            'chapter': csv_path.parents[1].name,
            'figure': csv_path.stem,
            'country_count': len(countries_found),
            'countries_found': '; '.join(countries_found),
            'rows': meta['rows'],
            'columns': meta['columns'],
            'column_names': meta['column_names'],
            'relative_path': csv_path.relative_to(repo_root).as_posix(),
        })

result_df = pd.DataFrame(records).sort_values(
    ['chapter', 'country_count'],
    ascending=[True, False]
).reset_index(drop=True)

print(f'Found {len(result_df)} datasets containing >= {MIN_COUNTRY_COUNT} target countries')
result_df.head(10)


Found 64 datasets containing >= 4 target countries


,chapter,figure,country_count,countries_found,rows,columns,column_names,relative_path
0,1. Research and Development,fig_1.1.3,8,United States; China; United Kingdom; France; ...,30,2,"Geographic area, Number of notable machine lea...",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
1,1. Research and Development,fig_1.3.1,8,United States; China; United Kingdom; France; ...,97,3,"Country, num_datacenters, iso_code",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
2,1. Research and Development,fig_1.3.2,8,United States; China; United Kingdom; France; ...,15,2,"Number of Data Centers, Country",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
3,1. Research and Development,fig_1.7.7,8,United States; China; United Kingdom; France; ...,31,3,"Country, Proximity_to_United_States, Proximity...",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
4,1. Research and Development,fig_1.8.1,7,United States; United Kingdom; France; Germany...,21,2,Number of top AI authors and inventors (in tho...,data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
5,1. Research and Development,fig_1.8.3,7,United States; United Kingdom; France; Germany...,1999,5,"year, country, country_code, education_label, ...",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
6,1. Research and Development,fig_1.8.4,7,United States; United Kingdom; France; Germany...,336,5,"year, country, country_code, Male, Female",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
7,1. Research and Development,fig_1.8.5,7,United States; United Kingdom; France; Germany...,14,22,"specialisation_area, Australia, Brazil, Canada...",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
8,1. Research and Development,fig_1.8.6,7,United States; United Kingdom; France; Germany...,3907,4,"country, country_code, year_month, netflow",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
9,1. Research and Development,fig_1.7.4,6,United States; China; United Kingdom; France; ...,15,2,"Granted AI patents (per 100,000 inhabitants), ...",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...


## 5. Summary by Chapter

In [5]:
chapter_summary = result_df.groupby('chapter').agg(
    dataset_count=('figure', 'count'),
    avg_countries=('country_count', 'mean'),
    max_countries=('country_count', 'max'),
).round(2)

chapter_summary


,dataset_count,avg_countries,max_countries
chapter,,,
1. Research and Development,13,6.85,8
3. Responsible AI,7,7.57,8
4. Economy,14,6.07,8
6. Medicine,1,7.00,7
7. Education,9,6.00,8
8. Policy and Governance,7,6.71,8
9. Public Opinion,13,7.08,8


## 6. Datasets Containing All 8 Countries

In [6]:
full_coverage = result_df[result_df['country_count'] == 8]
print(f'{len(full_coverage)} datasets contain all 8 target countries')
full_coverage[['chapter', 'figure', 'rows', 'columns', 'column_names']]


18 datasets contain all 8 target countries


,chapter,figure,rows,columns,column_names
0,1. Research and Development,fig_1.1.3,30,2,"Geographic area, Number of notable machine lea..."
1,1. Research and Development,fig_1.3.1,97,3,"Country, num_datacenters, iso_code"
2,1. Research and Development,fig_1.3.2,15,2,"Number of Data Centers, Country"
3,1. Research and Development,fig_1.7.7,31,3,"Country, Proximity_to_United_States, Proximity..."
13,3. Responsible AI,fig_3.6.1,138,2,"Country, Data Protection & Privacy"
14,3. Responsible AI,fig_3.6.2,176,3,"Country/Region, Country Code, Status_Category"
15,3. Responsible AI,fig_3.7.1,138,2,"Country, Bias & Unfair Discrimination"
16,3. Responsible AI,fig_3.7.2,138,2,"Country, Gender Equality"
17,3. Responsible AI,fig_3.7.3,138,2,"Country, Cultural & Linguistic Diversity"
20,4. Economy,fig_4.3.1,147,2,"Economy, H2 2025 AI Diffusion"


## 7. Save the Output

In [7]:
output_csv = repo_root / 'data' / 'number'/ 'g7_china_datasets.csv'
result_df.to_csv(output_csv, index=False)
print(f'Saved to: {output_csv}')
print(f'Total rows: {len(result_df)}')


Saved to: /home/wucheng/AI-Policy/data/number/g7_china_datasets.csv
Total rows: 64


## 8. Generate Markdown Summary

In [8]:
from data_utils import generate_markdown_summary

markdown_text = generate_markdown_summary(result_df, MIN_COUNTRY_COUNT)
output_md = repo_root / 'data' / 'g7_china_datasets_summary.md'
output_md.write_text(markdown_text, encoding='utf-8')
print(f'Saved to: {output_md}')



Saved to: /home/wucheng/AI-Policy/data/g7_china_datasets_summary.md


## 9. Preview Full Result

In [9]:
result_df


,chapter,figure,country_count,countries_found,rows,columns,column_names,relative_path
0,1. Research and Development,fig_1.1.3,8,United States; China; United Kingdom; France; ...,30,2,"Geographic area, Number of notable machine lea...",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
1,1. Research and Development,fig_1.3.1,8,United States; China; United Kingdom; France; ...,97,3,"Country, num_datacenters, iso_code",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
2,1. Research and Development,fig_1.3.2,8,United States; China; United Kingdom; France; ...,15,2,"Number of Data Centers, Country",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
3,1. Research and Development,fig_1.7.7,8,United States; China; United Kingdom; France; ...,31,3,"Country, Proximity_to_United_States, Proximity...",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
4,1. Research and Development,fig_1.8.1,7,United States; United Kingdom; France; Germany...,21,2,Number of top AI authors and inventors (in tho...,data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
...,...,...,...,...,...,...,...,...
59,9. Public Opinion,fig_9.1.8,7,United States; United Kingdom; France; Germany...,44,3,"Percent of respondents, Country, Label",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
60,9. Public Opinion,fig_9.1.9,7,United States; China; France; Germany; Italy; ...,224,3,"Category, Country, Response",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
61,9. Public Opinion,fig_9.2.10,7,United States; United Kingdom; France; Germany...,66,3,"Percent of respondents, Country, Label",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
62,9. Public Opinion,fig_9.3.3,7,United States; United Kingdom; France; Germany...,44,3,"Percent of respondents, Country, Label",data/number/PUBLIC DATA_ 2026 AI INDEX REPORT/...
